# Motor de Recomendación Estratégica en F1 (F1 Strategic Recommendation Engine)

Este notebook implementa y evalúa los dos modelos secuenciales que conforman el motor de recomendación de paradas en boxes (Capa 1: Regresión de Degradación Física y Capa 2: Ranking de Decisiones bajo Tráfico) desacoplados a través de una variable puente.

--- 
## Estructura del Notebook:
1. **Ingesta e Inferencia:** Carga de candidatos y cálculo del target físico (Capa 1).
2. **Capa 1 (Regresión de Degradación):** Evaluación comparativa inter-circuitos (GroupKFold) y entrenamiento del Stacking Regressor final.
3. **Capa Puente:** Cálculo del costo contrafáctico de permanencia en pista (`predicted_cost_of_staying`).
4. **Capa 2 (Ranking de Paradas):** Evaluación comparativa de baselines y modelos de Machine Learning (Point-wise y List-wise), y entrenamiento del RF Ranker final.
5. **Análisis de Errores (Error Analysis):** Evaluación y visualización detallada del recomendador en el GP de EUA.

In [5]:
import pandas as pd
import numpy as np
import os
import joblib
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

# Búsqueda robusta y dinámica del dataset
current_dir = Path(".").resolve()
DATA_PATH = None
PROJECT_DIR = None

for _ in range(6):
    # Intentar buscar project/data/recommendation
    path_opt1 = current_dir / "project" / "data" / "recommendation" / "pit_decision_candidates_v1.parquet"
    if path_opt1.exists():
        PROJECT_DIR = current_dir / "project"
        DATA_PATH = path_opt1
        break
    # Intentar buscar project/data/processed/recommendation
    path_opt1b = current_dir / "project" / "data" / "processed" / "recommendation" / "pit_decision_candidates_v1.parquet"
    if path_opt1b.exists():
        PROJECT_DIR = current_dir / "project"
        DATA_PATH = path_opt1b
        break
    # Intentar buscar data/recommendation
    path_opt2 = current_dir / "data" / "recommendation" / "pit_decision_candidates_v1.parquet"
    if path_opt2.exists():
        PROJECT_DIR = current_dir
        DATA_PATH = path_opt2
        break
    # Intentar buscar data/processed/recommendation
    path_opt2b = current_dir / "data" / "processed" / "recommendation" / "pit_decision_candidates_v1.parquet"
    if path_opt2b.exists():
        PROJECT_DIR = current_dir
        DATA_PATH = path_opt2b
        break
    current_dir = current_dir.parent

if DATA_PATH is None:
    raise FileNotFoundError("No se pudo localizar el archivo de candidatos pit_decision_candidates_v1.parquet en los directorios superiores. Coloca el archivo en project/data/recommendation o project/data/processed/recommendation, o ajusta la variable DATA_PATH.")

FEATURES_DIR = PROJECT_DIR / "data" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("Librerías importadas con éxito.")
print(f"Ruta del proyecto: {PROJECT_DIR.resolve()}")
print(f"Ruta del dataset: {DATA_PATH.resolve()}")
print(f"¿Existe el dataset?: {DATA_PATH.exists()}")

Librerías importadas con éxito.
Ruta del proyecto: D:\F1-data-project\project
Ruta del dataset: D:\F1-data-project\project\data\processed\recommendation\pit_decision_candidates_v1.parquet
¿Existe el dataset?: True


### 1. Ingesta de Datos y Target de la Capa 1
Calculamos el target `target_future_mean` para entrenar el modelo físico. Representa el ritmo esperado si se decide alargar la parada `wait_laps` vueltas.

In [6]:
def compute_regression_targets(df):
    """Calcula los objetivos de regresión para la Capa 1."""
    df = df.sort_values(["race_name", "driver_number", "lap_number"]).copy()
    lap_dur_dict = df.set_index(["race_name", "driver_number", "lap_number"])["lap_duration"].to_dict()
    stint_dict = df.set_index(["race_name", "driver_number", "lap_number"])["stint_number"].to_dict()
    pit_dict = df.set_index(["race_name", "driver_number", "lap_number"])["is_pit_lap"].to_dict()

    future_mean = []
    for idx, row in df.iterrows():
        race = row["race_name"]
        drv = row["driver_number"]
        lp = row["lap_number"]
        w = int(row["wait_laps"])

        if w == 0:
            future_mean.append(row["lap_duration"])
            continue

        laps_to_check = list(range(int(lp), int(lp) + w))
        durations = []
        valid = True
        stint_start = stint_dict.get((race, drv, lp))

        for curr_lp in laps_to_check:
            key = (race, drv, curr_lp)
            if key not in lap_dur_dict or stint_dict.get(key) != stint_start or (curr_lp > lp and pit_dict.get((race, drv, curr_lp), 0) == 1):
                valid = False
                break
            durations.append(lap_dur_dict[key])

        if valid and len(durations) == w:
            future_mean.append(np.mean(durations))
        else:
            future_mean.append(np.nan)

    df["target_future_mean"] = future_mean
    return df

df_raw = pd.read_parquet(DATA_PATH)
df_processed = compute_regression_targets(df_raw)
print(f"Dimensiones tras procesamiento: {df_processed.shape}")
print(f"Número de candidatos con target válido: {df_processed['target_future_mean'].notna().sum()}")

Dimensiones tras procesamiento: (19986, 27)
Número de candidatos con target válido: 18444


### 2. Capa 1: Modelado Comparativo de Regresión (Degradación del Neumático)
Evaluamos el rendimiento de la predicción de ritmo físico inter-circuitos mediante **GroupKFold (4 particiones)** agrupando por circuito (`race_name`).

In [ ]:
df_reg = df_processed.dropna(subset=["target_future_mean"]).copy()
race_means = df_reg.groupby("race_name")["target_future_mean"].transform("mean")
df_reg_clean = df_reg[df_reg["target_future_mean"] < race_means * 1.15].copy()

general_features = [
    "tyre_age", "compound_ord", "lap_vs_best_stint", "lap_mean_3", 
    "lap_std_3", "lap_slope_3", "deg_rate_3lap", "position", 
    "is_top10", "laps_remaining", "race_pct_complete", 
    "gap_ahead", "gap_behind", "wait_laps", "stint_number"
]

for col in general_features:
    median_val = df_reg_clean[col].median()
    df_reg_clean[col] = df_reg_clean[col].fillna(median_val)

X = df_reg_clean[general_features]
y = df_reg_clean["target_future_mean"]
race_groups = df_reg_clean["race_name"]

gkf = GroupKFold(n_splits=4)

models_to_compare = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
    "XGBoost Regressor": xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    "Extra Trees (Optimized)": ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=42),
    "Stacking Regressor (Ensamble Final)": StackingRegressor(
        estimators=[
            ('xgb', xgb.XGBRegressor(n_estimators=500, max_depth=8, learning_rate=0.05, random_state=42)),
            ('et', ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=42))
        ],
        final_estimator=Ridge()
    )
}

print("--- EVALUACIÓN COMPARATIVA CAPA 1 (CON OUTLIERS PARA COMPARACIÓN DIRECTA) ---")
for name, model in models_to_compare.items():
    mses_test, r2s_test = [], []
    mses_train, r2s_train = [], []
    for train_idx, test_idx in gkf.split(X, y, race_groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        preds_test = model.predict(X_test)
        preds_train = model.predict(X_train)
        
        mses_test.append(mean_squared_error(y_test, preds_test))
        r2s_test.append(r2_score(y_test, preds_test))
        mses_train.append(mean_squared_error(y_train, preds_train))
        r2s_train.append(r2_score(y_train, preds_train))
    
    print(f"Modelo: {name:<35} | MSE Test: {np.mean(mses_test):.4f} | R2 Test: {np.mean(r2s_test):.4f} | MSE Train: {np.mean(mses_train):.4f} | R2 Train: {np.mean(r2s_train):.4f}")

--- EVALUACIÓN COMPARATIVA CAPA 1 (CON OUTLIERS PARA COMPARACIÓN DIRECTA) ---
Modelo: Linear Regression                   | MSE Test: 73.7355 | R2 Test: -4.8686 | MSE Train: 9.6400 | R2 Train: 0.6811
Modelo: Decision Tree                       | MSE Test: 30.6726 | R2 Test: -1.4678 | MSE Train: 4.0088 | R2 Train: 0.8637


#### Entrenamiento del Stacking Regressor de Producción
Entrenamos el ensamble final con todas las variables, incluyendo variables categóricas del circuito (dummy) y el identificador del piloto, para exportar el modelo.

In [ ]:
df_reg_final = pd.get_dummies(df_reg_clean, columns=["race_name"])
features_final = [
    "tyre_age", "compound_ord", "lap_vs_best_stint", "lap_mean_3", 
    "lap_std_3", "lap_slope_3", "deg_rate_3lap", "position", 
    "is_top10", "laps_remaining", "race_pct_complete", 
    "gap_ahead", "gap_behind", "wait_laps", "stint_number", "driver_number"
]
race_features = [col for col in df_reg_final.columns if col.startswith("race_name_")]
features_final += race_features

for col in features_final:
    median_val = df_reg_final[col].median()
    df_reg_final[col] = df_reg_final[col].fillna(median_val if not pd.isna(median_val) else 0.0)

X_final = df_reg_final[features_final]
y_final = df_reg_final["target_future_mean"]

estimators_prod = [
    ('xgb', xgb.XGBRegressor(n_estimators=500, max_depth=8, learning_rate=0.05, random_state=42)),
    ('et', ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=42))
]
production_stacking = StackingRegressor(estimators=estimators_prod, final_estimator=Ridge())

print(f"Entrenando Stacking definitivo en {len(X_final)} registros...")
production_stacking.fit(X_final, y_final)

joblib.dump(production_stacking, FEATURES_DIR / "regression_layer1_model.pkl")
joblib.dump(features_final, FEATURES_DIR / "regression_features.joblib")

preds_train = production_stacking.predict(X_final)
print("Capa 1 entrenada y exportada con éxito.")
print(f"R2 Train Score final: {r2_score(y_final, preds_train):.4f}")
print(f"MSE Train Score final: {mean_squared_error(y_final, preds_train):.4f}")

Entrenando Stacking definitivo en 16746 registros...


Capa 1 entrenada y exportada con éxito.
R2 Train Score final: 0.9941
MSE Train Score final: 0.2096


### 3. Capa Puente: Cálculo de Costo de Permanencia en Pista
Utilizamos el modelo entrenado de la Capa 1 para estimar la pérdida de tiempo contrafáctica si el monoplaza continúa en pista, calculando la característica puente `predicted_cost_of_staying`.

In [ ]:
reg_features = joblib.load(FEATURES_DIR / "regression_features.joblib")
reg_model = joblib.load(FEATURES_DIR / "regression_layer1_model.pkl")

df_all = df_processed.copy()
df_all_dummies = pd.get_dummies(df_all, columns=["race_name"])

for col in reg_features:
    if col not in df_all_dummies.columns:
        df_all_dummies[col] = 0.0
    median_val = df_all_dummies[col].median()
    df_all_dummies[col] = df_all_dummies[col].fillna(median_val if not pd.isna(median_val) else 0.0)

df_all["predicted_future_pace"] = reg_model.predict(df_all_dummies[reg_features])
df_all["predicted_cost_of_staying"] = df_all["wait_laps"] * (df_all["predicted_future_pace"] - df_all["lap_duration"])

df_all.to_parquet(DATA_PATH, index=False)
print("Costo de permanencia calculado y guardado con éxito.")

Costo de permanencia calculado y guardado con éxito.


### 4. Capa 2: Ranking y Recomendación de Paradas (Modelos y Baselines)
Evaluamos y comparamos modelos de Ranking (Random Forest Regressor Point-wise y XGBRanker) frente a tres baselines de referencia (Random, Heurística de Edad de Neumático y Popularidad Empírica) bajo **GroupKFold (4 splits por circuito)**.

In [ ]:
def evaluate_ndcg(df_eval, group_cols, rank_col, label_col, k=3):
    ndcgs = []
    for _, group in df_eval.groupby(group_cols):
        if len(group) < 2:
            continue
        sorted_group = group.sort_values(by=rank_col, ascending=False)
        actual_labels = sorted_group[label_col].values
        
        if np.all(actual_labels == actual_labels[0]):
            continue
            
        ideal_labels = np.sort(group[label_col].values)[::-1]
        
        dcg, idcg = 0.0, 0.0
        for i in range(min(k, len(actual_labels))):
            rel = actual_labels[i]
            rel_norm = max(0, rel + 2)
            dcg += (2**rel_norm - 1) / np.log2(i + 2)
            
            ideal_rel = ideal_labels[i]
            ideal_rel_norm = max(0, ideal_rel + 2)
            idcg += (2**ideal_rel_norm - 1) / np.log2(i + 2)
            
        if idcg > 0:
            ndcgs.append(dcg / idcg)
            
    return np.mean(ndcgs) if ndcgs else 1.0


df_rank = df_all.copy()
df_rank["query_id"] = df_rank["race_name"] + "_" + df_rank["driver_number"].astype(str) + "_" + df_rank["lap_number"].astype(str)
df_rank = df_rank.sort_values("query_id")
df_rank["rank_label"] = df_rank.groupby("query_id")["success_score_label"].rank(method="first").astype(int) - 1

ranking_features = [
    "tyre_age", "compound_ord", "lap_vs_best_stint", "lap_mean_3", 
    "lap_std_3", "lap_slope_3", "deg_rate_3lap", "position", 
    "is_top10", "laps_remaining", "race_pct_complete", 
    "gap_ahead", "gap_behind", "wait_laps", "stint_number", "predicted_cost_of_staying"
]

for col in ranking_features:
    median_val = df_rank[col].median()
    df_rank[col] = df_rank[col].fillna(median_val if not pd.isna(median_val) else 0.0)

X_rank = df_rank[ranking_features]
y_rank = df_rank["success_score_label"]
races_groups = df_rank["race_name"]

metrics_log = {m: [] for m in ["rand1", "rand3", "heur1", "heur3", "pop1", "pop3", "rf1", "rf3", "xgb1", "xgb3"]}
gkf_rank = GroupKFold(n_splits=4)

for train_idx, test_idx in gkf_rank.split(X_rank, y_rank, races_groups):
    df_te_split = df_rank.iloc[test_idx].copy()
    df_tr_split = df_rank.iloc[train_idx].copy()
    
    # 1. Random Baseline
    np.random.seed(42)
    df_te_split["pred_random"] = np.random.rand(len(df_te_split))
    
    # 2. Heuristic Baseline (Tyre-Age 18)
    df_te_split["pred_heuristic"] = -np.abs((df_te_split["tyre_age"] + df_te_split["wait_laps"]) - 18)
    
    # 3. Popularity Baseline
    pit_counts = df_tr_split[df_tr_split["is_pit_lap"] == 1].groupby(["compound_ord", "tyre_age"]).size().to_dict()
    tot_counts = df_tr_split.groupby(["compound_ord", "tyre_age"]).size().to_dict()
    
    def get_pop_score(row):
        proj_age = row["tyre_age"] + row["wait_laps"]
        comp = row["compound_ord"]
        numerator = pit_counts.get((comp, proj_age), 0)
        denominator = tot_counts.get((comp, proj_age), 1)
        return numerator / denominator
        
    df_te_split["pred_popularity"] = df_te_split.apply(get_pop_score, axis=1)
    
    # 4. Point-wise RF Ranker
    rf_ranker = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
    rf_ranker.fit(df_tr_split[ranking_features], df_tr_split["success_score_label"])
    df_te_split["pred_rf"] = rf_ranker.predict(df_te_split[ranking_features])
    
    # 5. List-wise XGBRanker
    q_tr = df_tr_split.groupby("query_id").size().values
    q_te = df_te_split.groupby("query_id").size().values
    xgbr = xgb.XGBRanker(n_estimators=100, max_depth=5, learning_rate=0.1, objective="rank:ndcg", random_state=42)
    xgbr.fit(df_tr_split[ranking_features], df_tr_split["rank_label"], group=q_tr)
    df_te_split["pred_xgb"] = xgbr.predict(df_te_split[ranking_features])
    
    # Evaluar NDCG
    metrics_log["rand1"].append(evaluate_ndcg(df_te_split, "query_id", "pred_random", "success_score_label", k=1))
    metrics_log["rand3"].append(evaluate_ndcg(df_te_split, "query_id", "pred_random", "success_score_label", k=3))
    metrics_log["heur1"].append(evaluate_ndcg(df_te_split, "query_id", "pred_heuristic", "success_score_label", k=1))
    metrics_log["heur3"].append(evaluate_ndcg(df_te_split, "query_id", "pred_heuristic", "success_score_label", k=3))
    metrics_log["pop1"].append(evaluate_ndcg(df_te_split, "query_id", "pred_popularity", "success_score_label", k=1))
    metrics_log["pop3"].append(evaluate_ndcg(df_te_split, "query_id", "pred_popularity", "success_score_label", k=3))
    metrics_log["rf1"].append(evaluate_ndcg(df_te_split, "query_id", "pred_rf", "success_score_label", k=1))
    metrics_log["rf3"].append(evaluate_ndcg(df_te_split, "query_id", "pred_rf", "success_score_label", k=3))
    metrics_log["xgb1"].append(evaluate_ndcg(df_te_split, "query_id", "pred_xgb", "success_score_label", k=1))
    metrics_log["xgb3"].append(evaluate_ndcg(df_te_split, "query_id", "pred_xgb", "success_score_label", k=3))

print("\n--- RESULTADOS OFFLINE: COMPARATIVO DE RANKING ---")
print(f"{'Modelo / Sistema':<30} | {'NDCG@1 Promedio':<15} | {'NDCG@3 Promedio':<15}")
print("-" * 66)
print(f"{'Random Baseline':<30} | {np.mean(metrics_log['rand1']):.4f}          | {np.mean(metrics_log['rand3']):.4f}")
print(f"{'Tyre-Age Heuristic (18L)':<30} | {np.mean(metrics_log['heur1']):.4f}          | {np.mean(metrics_log['heur3']):.4f}")
print(f"{'Popularity Baseline':<30} | {np.mean(metrics_log['pop1']):.4f}          | {np.mean(metrics_log['pop3']):.4f}")
print(f"{'Random Forest Point-wise (Selected)':<30} | {np.mean(metrics_log['rf1']):.4f}          | {np.mean(metrics_log['rf3']):.4f}")
print(f"{'XGBRanker List-wise':<30} | {np.mean(metrics_log['xgb1']):.4f}          | {np.mean(metrics_log['xgb3']):.4f}")


--- RESULTADOS OFFLINE: COMPARATIVO DE RANKING ---
Modelo / Sistema               | NDCG@1 Promedio | NDCG@3 Promedio
------------------------------------------------------------------
Random Baseline                | 0.3802          | 0.5212
Tyre-Age Heuristic (18L)       | 0.4605          | 0.4782
Popularity Baseline            | 0.5553          | 0.6577
Random Forest Point-wise (Selected) | 0.8901          | 0.9135
XGBRanker List-wise            | 0.9248          | 0.9512


#### Entrenamiento del RF Ranker Definitivo (Capa 2)
Entrenamos el clasificador de producción sobre todos los datos.

In [ ]:
final_rf_ranker = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
final_rf_ranker.fit(X_rank, y_rank)

joblib.dump(final_rf_ranker, FEATURES_DIR / "ranking_layer2_model.pkl")
print("Modelo Capa 2 de producción exportado con éxito.")

Modelo Capa 2 de producción exportado con éxito.


### 5. Análisis de Errores (Error Analysis) en el GP de EUA
Para validar el recomendador en un escenario inter-circuito realista, realizamos inferencia sobre el GP de Estados Unidos (habiendo entrenado los modelos únicamente en Australia, Japón y China) y analizamos detalladamente los casos fuertes y de falla del recomendador.

In [ ]:
df_train_splits = df_rank[df_rank["race_name"] != "united_states"].copy()
df_test_split = df_rank[df_rank["race_name"] == "united_states"].copy()

# 1. Entrenar Capa 1 de prueba
df_reg_train = df_reg_final[~df_reg_final["race_name_united_states"].fillna(0.0).astype(bool)].copy()
X_reg_train = df_reg_train[features_final]
y_reg_train = df_reg_train["target_future_mean"]
eval_stacking = StackingRegressor(estimators=estimators_prod, final_estimator=Ridge())
eval_stacking.fit(X_reg_train, y_reg_train)

# 2. Estimar costo puente sobre USA
df_test_dummies = pd.get_dummies(df_test_split, columns=["race_name"])
for col in features_final:
    if col not in df_test_dummies.columns:
        df_test_dummies[col] = 0.0
df_test_split["predicted_future_pace"] = eval_stacking.predict(df_test_dummies[features_final])
df_test_split["predicted_cost_of_staying"] = df_test_split["wait_laps"] * (df_test_split["predicted_future_pace"] - df_test_split["lap_duration"])

# 3. Entrenar Ranker Capa 2 de prueba
eval_rf_ranker = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
eval_rf_ranker.fit(df_train_splits[ranking_features], df_train_splits["success_score_label"])
df_test_split["pred_rf_score"] = eval_rf_ranker.predict(df_test_split[ranking_features])

# 4. Porcentaje de coincidencia
queries_test = df_test_split.groupby("query_id")
hits, total_queries = 0, 0
for q_id, group in queries_test:
    if len(group) < 2:
        continue
    best_pred_idx = group["pred_rf_score"].idxmax()
    best_real_idx = group["success_score_label"].idxmax()
    if group.loc[best_pred_idx, "wait_laps"] == group.loc[best_real_idx, "wait_laps"]:
        hits += 1
    total_queries += 1
print(f"Exactitud de recomendación en GP de EUA (Test): {hits/total_queries*100:.2f}% ({hits}/{total_queries})")

# 5. Mostrar ejemplos específicos
print("\n--- CASOS FUERTES (EJEMPLOS DE ACIERTOS CLAVE) ---")
q_ver = df_test_split[(df_test_split["driver_number"] == 1.0) & (df_test_split["lap_number"] == 1.0)]
if not q_ver.empty:
    print("\nMax Verstappen, Vuelta 1 (GP de EUA):")
    print(q_ver[["wait_laps", "predicted_cost_of_staying", "success_score_label", "pred_rf_score"]].to_string(index=False))

q_ham = df_test_split[(df_test_split["driver_number"] == 44.0) & (df_test_split["lap_number"] == 35.0)]
if not q_ham.empty:
    print("\nLewis Hamilton, Vuelta 35 (GP de EUA):")
    print(q_ham[["wait_laps", "predicted_cost_of_staying", "success_score_label", "pred_rf_score"]].to_string(index=False))

print("\n--- CASOS DE FALLA (DISCREPANCIAS TÁCTICAS) ---")
q_hul = df_test_split[(df_test_split["driver_number"] == 27.0) & (df_test_split["lap_number"] == 1.0)]
if not q_hul.empty:
    print("\nNico Hülkenberg, Vuelta 1 (GP de EUA) - Choque en largada:")
    print(q_hul[["wait_laps", "predicted_cost_of_staying", "success_score_label", "pred_rf_score"]].to_string(index=False))

q_ver39 = df_test_split[(df_test_split["driver_number"] == 1.0) & (df_test_split["lap_number"] == 39.0)]
if not q_ver39.empty:
    print("\nMax Verstappen, Vuelta 39 (GP de EUA) - Undercut defensivo en Hard:")
    print(q_ver39[["wait_laps", "predicted_cost_of_staying", "success_score_label", "pred_rf_score"]].to_string(index=False))

Exactitud de recomendación en GP de EUA (Test): 90.06% (915/1016)

--- CASOS FUERTES (EJEMPLOS DE ACIERTOS CLAVE) ---

Max Verstappen, Vuelta 1 (GP de EUA):
 wait_laps  predicted_cost_of_staying  success_score_label  pred_rf_score
         0                   0.000000                  0.0       0.759413
         1                   2.899833                 -2.0      -2.113745
         2                   6.284876                 -2.0      -0.811060
         3                   9.632133                 -2.0      -1.263250
         4                  15.746771                 -2.0      -0.966460
         5                  23.600156                 -2.0      -0.943514

Lewis Hamilton, Vuelta 35 (GP de EUA):
 wait_laps  predicted_cost_of_staying  success_score_label  pred_rf_score
         0                   0.000000                  0.0      -0.005954
         1                   1.707520                 -2.0      -1.979017
         3                   5.428191                 -2.0     